# Bayes' Theorem
**Topic:** Probability & Statistics

In [4]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** the difference between a prior belief and a posterior belief after seeing evidence
- **Calculate** a posterior probability using Bayes' theorem from a prior, likelihood, and marginal
- **Interpret** why a high-accuracy test can still produce mostly false positives in a low-prevalence setting

> **Tip:** Start by moving the **prior probability slider** to very low values like 1%, watch how the posterior barely budges even with strong evidence, and then move it to 50% to see the dramatic difference.

---
## How we got here

In *02: Conditional Probability & Independence* we learned to compute P(A | B), the probability of A given that B occurred. But sometimes we know the "wrong" direction of the conditional and need to flip it. A doctor knows P(positive test | disease), but needs P(disease | positive test). Bayes' theorem is the formula that performs that flip.

---
## Why this matters for data science

Bayes' theorem underpins an entire family of machine learning methods, Naive Bayes classifiers, Bayesian neural networks, and probabilistic graphical models all start here. More broadly, the Bayesian *mindset* of maintaining and updating uncertainty is increasingly how modern ML systems are designed, from uncertainty quantification to A/B test analysis.

---
## Try it yourself

In [5]:
from scipy.optimize import brentq
from tkh_utils import make_slider, make_output, display_widget, register_observer

# ── Circle geometry (same pattern as 02) ──────────────────────────────────────
_RMAX = 0.34

def _int_area(d, r1, r2):
    if d >= r1 + r2: return 0.0
    if d <= abs(r1 - r2): return np.pi * min(r1, r2)**2
    a1 = np.arccos(np.clip((d**2 + r1**2 - r2**2) / (2*d*r1), -1, 1))
    a2 = np.arccos(np.clip((d**2 + r2**2 - r1**2) / (2*d*r2), -1, 1))
    sq = np.sqrt(max((-d+r1+r2)*(d+r1-r2)*(d-r1+r2)*(d+r1+r2), 0))
    return r1**2 * a1 + r2**2 * a2 - 0.5 * sq

def _solve_d(r1, r2, target):
    m = np.pi * min(r1, r2)**2
    if target <= 0: return r1 + r2 + 0.01
    if target >= m - 1e-9: return max(abs(r1 - r2) - 0.005, 1e-4)
    try:
        return brentq(lambda d: _int_area(d, r1, r2) - target,
                      max(abs(r1 - r2) + 1e-9, 1e-9), r1 + r2 - 1e-9)
    except Exception:
        return (r1 + r2) / 2

def _circle(cx, cy, r, n=200):
    t = np.linspace(0, 2*np.pi, n)
    return cx + r*np.cos(t), cy + r*np.sin(t)

def _intersect(cx1, cy1, r1, cx2, cy2, r2):
    d = np.hypot(cx2 - cx1, cy2 - cy1)
    if d >= r1 + r2 - 1e-9 or d < 1e-10: return None, None
    if d <= abs(r1 - r2) + 1e-9:
        r = min(r1, r2)
        cx, cy_c = (cx1, cy1) if r1 <= r2 else (cx2, cy2)
        return _circle(cx, cy_c, r)
    ang = np.arctan2(cy2 - cy1, cx2 - cx1)
    a1  = np.arccos(np.clip((d**2 + r1**2 - r2**2) / (2*d*r1), -1, 1))
    a2  = np.arccos(np.clip((d**2 + r2**2 - r1**2) / (2*d*r2), -1, 1))
    t1  = np.linspace(ang - a1, ang + a1, 80)
    t2  = np.linspace(ang + np.pi + a2, ang + np.pi - a2, 80)
    xp  = np.concatenate([cx1 + r1*np.cos(t1), cx2 + r2*np.cos(t2), [cx1 + r1*np.cos(t1[0])]])
    yp  = np.concatenate([cy1 + r1*np.sin(t1), cy2 + r2*np.sin(t2), [cy1 + r1*np.sin(t1[0])]])
    return xp, yp

# ── Controls ──────────────────────────────────────────────────────────────────
out          = make_output()
prior_slider = make_slider(0.01, 0.99, 0.01, 0.02, "Prior P(H):")
sens_slider  = make_slider(0.50, 0.99, 0.01, 0.95, "Sensitivity P(E|H):")
fpr_slider   = make_slider(0.01, 0.50, 0.01, 0.05, "False pos. rate P(E|¬H):")

# ── Render ────────────────────────────────────────────────────────────────────
def render(p_h, p_eh, p_e_nh):
    p_nh = 1.0 - p_h
    p_he = p_eh  * p_h    # P(H ∩ E)  — true positives
    p_fp = p_e_nh * p_nh  # P(¬H ∩ E) — false positives
    p_e  = p_he + p_fp    # P(E)       — marginal evidence probability
    post = p_he / p_e if p_e > 1e-9 else 0.0  # P(H|E) — posterior

    r_h  = _RMAX * np.sqrt(p_h)
    r_e  = _RMAX * np.sqrt(p_e)
    d    = _solve_d(r_h, r_e, p_he * np.pi * _RMAX**2)
    ch, ce, cy = 0.5 - d/2, 0.5 + d/2, 0.5

    traces = []

    # E circle: filled (conditioning event — "given we observed E")
    ex, ey = _circle(ce, cy, r_e)
    traces.append(go.Scatter(x=ex, y=ey, fill="toself",
        fillcolor="rgba(180,215,180,0.52)",
        line=dict(color=PALETTE["secondary"], width=2.5),
        name="E (evidence)", hoverinfo="skip"))

    # H circle: outline only (what fraction of E is also H?)
    hx, hy = _circle(ch, cy, r_h)
    traces.append(go.Scatter(x=hx, y=hy, fill="none",
        line=dict(color=PALETTE["primary"], width=2.5),
        name="H (hypothesis)", hoverinfo="skip"))

    # H ∩ E: true positives — darker fill
    ix, iy = _intersect(ch, cy, r_h, ce, cy, r_e)
    if ix is not None:
        traces.append(go.Scatter(x=ix, y=iy, fill="toself",
            fillcolor="rgba(70,115,70,0.75)",
            line=dict(width=0), name="H ∩ E  (true positives)", hoverinfo="skip"))
        mid = (ch + ce) / 2
        traces.append(go.Scatter(x=[mid], y=[cy], mode="markers",
            marker=dict(color="black", size=7),
            showlegend=False, hoverinfo="skip"))

    # ── Annotations ───────────────────────────────────────────────────────────
    anns = [
        # H and E circle labels
        dict(x=ch - r_h*0.55, y=cy + r_h*0.80, text="<i>H</i>",
             showarrow=False, font=dict(size=20, color=PALETTE["primary"]),
             xanchor="center"),
        dict(x=ce + r_e*0.55, y=cy + r_e*0.80, text="<i>E</i>",
             showarrow=False, font=dict(size=20, color=PALETTE["secondary"]),
             xanchor="center"),
        # Sample space label
        dict(x=1.04, y=1.07, text="<i>S</i>",
             showarrow=False, font=dict(size=16, color=PALETTE["muted"]),
             xanchor="right"),
    ]

    # Arrow annotation to H ∩ E
    if ix is not None:
        mid = (ch + ce) / 2
        anns.append(dict(
            x=mid, y=cy,
            ax=mid + 0.20, ay=cy - 0.26,
            xref="x", yref="y", axref="x", ayref="y",
            text=f"<i>H</i> ∩ <i>E</i>  =  {p_he:.4f}",
            showarrow=True, arrowhead=2, arrowsize=1.2, arrowwidth=1.5,
            font=dict(size=12, color="#333"),
        ))

    # False positive label in E-only region (right portion of E circle)
    fp_x = ce + r_e * 0.42
    if fp_x > ch + r_h + 0.04 and p_fp > 0.005:
        anns.append(dict(
            x=fp_x, y=cy - r_e * 0.18,
            text=f"FP = {p_fp:.4f}",
            showarrow=False,
            font=dict(size=11, color="#555"),
            xanchor="center",
        ))

    # Posterior callout: P(H|E) = dark region / full E circle
    anns.append(dict(
        x=0.50, y=-0.10, xref="x", yref="y",
        text=(f"P(H|E) = <b>{post:.1%}</b>   "
              f"← the dark region is {post:.1%} of the <i>E</i> circle"),
        showarrow=False,
        font=dict(size=13, color=PALETTE["accent"]),
        xanchor="center",
    ))

    title = (
        f"P(H) = {p_h:.2f}   ·   P(E|H) = {p_eh:.2f}   ·   P(E|¬H) = {p_e_nh:.2f}<br>"
        f"P(H|E) = P(H∩E) / P(E) = {p_he:.4f} / {p_e:.4f} = <b>{post:.1%}</b>"
    )

    layout = base_layout(title=title)
    layout.update(
        width=560, height=540,
        xaxis=dict(range=[-0.15, 1.15], showgrid=False, zeroline=False,
                   showticklabels=False, showline=False),
        yaxis=dict(range=[-0.18, 1.12], showgrid=False, zeroline=False,
                   showticklabels=False, showline=False,
                   scaleanchor="x", scaleratio=1),
        shapes=[dict(type="rect", x0=-0.05, y0=-0.05, x1=1.05, y1=1.05,
                     fillcolor="rgba(245,245,245,0.50)",
                     line=dict(color=PALETTE["muted"], width=1.5))],
        annotations=anns,
        showlegend=False,
        margin=dict(l=20, r=20, t=95, b=20),
    )
    fig = go.Figure(data=traces, layout=layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

# ── Observers ─────────────────────────────────────────────────────────────────
register_observer(
    [prior_slider, sens_slider, fpr_slider],
    lambda change: render(prior_slider.value, sens_slider.value, fpr_slider.value),
)
display_widget(VBox([prior_slider, sens_slider, fpr_slider]), out)
render(prior_slider.value, sens_slider.value, fpr_slider.value)


---
## What's happening?

Bayes' theorem is a formula for updating a belief when new evidence arrives. We start with a **prior**, what we believed before seeing the data, and combine it with the **likelihood** of the evidence under our hypothesis to get a **posterior**, our updated belief.

| Symbol | What it controls |
|--------|-----------------|
| P(H) | Prior probability of the hypothesis |
| P(E \| H) | Likelihood: how probable is the evidence if H is true? |
| P(E) | Marginal probability of evidence (normalizing constant) |
| P(H \| E) | Posterior: updated probability after seeing evidence |

$$P(H \mid E) = \frac{P(E \mid H) \cdot P(H)}{P(E)}$$

where the denominator expands as:

$$P(E) = P(E \mid H)\,P(H) + P(E \mid \neg H)\,P(\neg H)$$

### The base rate matters more than you think

When the prior P(H) is very small, even strong evidence only moves the posterior a little. Go back to the widget and set the prior to 1% with a 99% accurate test, the posterior is still below 50%. This is not a flaw in the test; it's mathematics.

---
## Real-world example: Spam filter classification

A spam filter is a live Bayes' theorem machine. It starts with a prior P(spam) based on how much spam typically flows into inboxes, then updates that probability as it reads each word in the email.

The chart below simulates how the posterior probability of spam updates as a Naive Bayes filter reads successive words in an email containing "free", "winner", "click", and "unsubscribe". Notice:

- **Notice:** The prior starts near 45%, close to half of all email is spam, and the first suspicious word immediately pushes the posterior above 70%
- **Notice:** Each additional keyword compounds the evidence; by the fourth word the model is over 97% confident the email is spam
- **Notice:** The update slows as it approaches 100%, it becomes increasingly hard to move a probability that's already very high

> **Discussion question:** If the filter misclassifies a legitimate newsletter as spam (a false positive), what would you adjust, the prior, the likelihood, or the decision threshold, and why?

In [6]:
# ── Naive Bayes spam filter simulation ─────────────────────────────────────────
# P(word | spam) values estimated from common spam-filtering research
words        = ["(prior)", "free", "winner", "click", "unsubscribe"]
p_word_spam  = [None, 0.80, 0.72, 0.65, 0.60]  # P(word|spam)
p_word_ham   = [None, 0.10, 0.05, 0.20, 0.30]  # P(word|not spam)

prior_spam = 0.45
posteriors = [prior_spam]
p = prior_spam

for i in range(1, len(words)):
    ps  = p_word_spam[i]
    ph  = p_word_ham[i]
    # Bayes update
    numerator   = ps * p
    denominator = ps * p + ph * (1 - p)
    p = numerator / denominator
    posteriors.append(p)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=words, y=posteriors,
    mode="lines+markers",
    line=dict(color=PALETTE["primary"], width=3),
    marker=dict(size=10, color=PALETTE["secondary"]),
    name="P(spam | words seen so far)",
    text=[f"{v:.1%}" for v in posteriors],
    textposition="top center",
))
fig.add_hline(y=0.5, line_dash="dot", line_color=PALETTE["muted"],
              annotation_text="Decision threshold (50%)")
layout = base_layout(
    title="Bayesian Spam Filter — Posterior Updates Word by Word",
    xaxis_title="Word read (cumulative evidence)",
    yaxis_title="P(spam | evidence so far)",
)
layout.update(yaxis=dict(range=[0, 1.1], tickformat=".0%"))
fig.update_layout(layout)
fig.show()

### Other everyday Bayes' theorem applications

| Application | Prior P(H) | Evidence E | Posterior P(H \| E) direction |
|-------------|-----------|-----------|-------------------------------|
| Spam filter | 45% of email is spam | Email contains "free money" | Rises sharply toward 97%+ |
| Medical screening | Disease prevalence in population | Positive test result | Rises, but base rate limits it |
| Credit risk | Default rate for loan category | Missed payment last month | Rises based on historical rates |
| Search ranking | Document relevance prior | User clicked result | Rises: click is evidence of relevance |
| Autonomous vehicles | Object is a pedestrian | Lidar shape matches human | Updated per sensor reading |

---
## Key takeaway

> **Bayes' theorem flips a conditional probability by weighting the likelihood with the prior; when the prior is small, even powerful evidence cannot fully overcome the rarity of the hypothesis.**

---
*Next up: Central Tendency — moving from probability to the first summary statistics that describe where data lives*